<a href="https://colab.research.google.com/github/EliasNoorzad/temporal-reasoning-TISER/blob/main/notebooks/run_test_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/EliasNoorzad/temporal-reasoning-TISER.git

Cloning into 'temporal-reasoning-TISER'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 323 (delta 52), reused 85 (delta 27), pack-reused 207 (from 1)
Receiving objects: 100% (323/323), 456.14 KiB | 3.54 MiB/s, done.
Resolving deltas: 100% (181/181), done.


In [2]:
%cd temporal-reasoning-TISER
!ls

/content/temporal-reasoning-TISER
extensions  notebooks  README.md  requirements.txt  src


In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.6 MB/s eta 0:00:00


In [5]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
import gc

from argparse import Namespace
from pathlib import Path
import torch

from src.dataset import load_filtered_test_dataset

from src.evaluate import (
    load_evaluation_model,
    run_combined_prompt_evaluation,
    read_jsonl,
    write_combined_summary,
)

LORA_ADAPTER_PATH = "EliElias/TISER-Qwen2.5-3B-LoRA"

OUTPUT_DIR = Path("/content/evaluation_outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_dataset = load_filtered_test_dataset()

print(f"Filtered test examples: {len(test_dataset)}")
print(f"Results will be saved to: {OUTPUT_DIR}")

README.md:   0%|          | 0.00/4.00k [00:00<?, ?B/s]

data/TISER_train.json: reconstructing file:   0%|          |  0.00B /  208MB            

data/TISER_train.json: downloading bytes:           |  0.00B            

data/TISER_test.json: reconstructing file:   0%|          |  0.00B / 88.0MB            

data/TISER_test.json: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Original test examples: 22014
Test examples <= 2048 tokens: 20442
Removed examples: 1572
Filtered test examples: 20442
Results will be saved to: /content/evaluation_outputs


In [7]:
def run_full_test_both(
    model_type="lora",
    lora_adapter_path=None,
    batch_size=16,
):
    args = Namespace(
        model_type=model_type,
        prompt_type="both",
        lora_adapter_path=lora_adapter_path,
        output_dir=str(OUTPUT_DIR),
        direct_max_new_tokens=128,
        tiser_max_new_tokens=2048,
        batch_size=batch_size,
        device_map="auto",
    )

    tokenizer, model = load_evaluation_model(args)

    results_path = OUTPUT_DIR / f"{model_type}_both_results.jsonl"
    summary_path = OUTPUT_DIR / f"{model_type}_both_summary.json"

    run_combined_prompt_evaluation(
        args=args,
        model=model,
        tokenizer=tokenizer,
        test_dataset=test_dataset,
        results_path=results_path,
    )

    records = read_jsonl(results_path)
    summary = write_combined_summary(summary_path, records)

    print("\n==============================")
    print("LORA — DIRECT + TISER FULL TEST")
    print("==============================")
    print(f"Examples: {len(records)}")

    print(f"Direct EM: {summary['direct_overall_em']:.2f}%")
    print(f"Direct F1: {summary['direct_overall_token_f1']:.2f}%")
    print(
        f"Direct avg generated tokens: "
        f"{summary['direct_average_generated_tokens']:.2f}"
    )

    print(f"TISER EM: {summary['tiser_overall_em']:.2f}%")
    print(f"TISER F1: {summary['tiser_overall_token_f1']:.2f}%")
    print(
        f"TISER avg generated tokens: "
        f"{summary['tiser_average_generated_tokens']:.2f}"
    )

    print(f"Saved: {results_path}")
    print(f"Saved: {summary_path}")

    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

In [ ]:
lora_both_summary = run_full_test_both(
    model_type="lora",
    lora_adapter_path=LORA_ADAPTER_PATH,
    batch_size=16,
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generating direct and TISER:   0%|          | 0/20442 [00:00<?, ?it/s]


LORA — DIRECT + TISER FULL TEST
Examples: 20442
Direct EM: 74.83%
Direct F1: 81.27%
Direct avg generated tokens: 5.96
TISER EM: 80.42%
TISER F1: 85.91%
TISER avg generated tokens: 283.00
Saved: /content/evaluation_outputs/lora_both_results.jsonl
Saved: /content/evaluation_outputs/lora_both_summary.json


In [7]:
from huggingface_hub import hf_hub_download

INPUT_PATH = hf_hub_download(
    repo_id="EliElias/TISER-Evaluation-Results",
    filename="lora_both_results.jsonl",
    repo_type="dataset",
)

!python src/evaluate.py \
    --rescore-existing "{INPUT_PATH}" \
    --output-rescored-results "{OUTPUT_DIR}/lora_both_results_rescored.jsonl" \
    --output-summary "{OUTPUT_DIR}/lora_both_summary_rescored.json"

lora_both_results.jsonl: reconstructing file:   0%|          |  0.00B / 47.2MB            

lora_both_results.jsonl: downloading bytes:           |  0.00B            

Rescored existing predictions: /root/.cache/huggingface/hub/datasets--EliElias--TISER-Evaluation-Results/snapshots/79c787efd5969210f0e5e3b701f4d414854c135b/lora_both_results.jsonl
Saved rescored predictions to: /content/evaluation_outputs/lora_both_results_rescored.jsonl
Saved corrected summary to: /content/evaluation_outputs/lora_both_summary_rescored.json
Direct five-dataset Macro EM: 79.07%
Direct five-dataset Macro F1: 85.54%
Direct in-domain generated tokens: average 5.93, total 113257
TISER five-dataset Macro EM: 86.14%
TISER five-dataset Macro F1: 90.73%
TISER in-domain generated tokens: average 272.00, total 5195747


In [8]:
import json

with open(OUTPUT_DIR / "lora_both_summary_rescored.json", "r") as f:
    summary = json.load(f)

for dataset, metrics in summary["in_domain_per_dataset"].items():
    print(
        dataset,
        f"Direct EM: {metrics['direct_em']:.2f}",
        f"Direct F1: {metrics['direct_f1']:.2f}",
        f"TISER EM: {metrics['tiser_em']:.2f}",
        f"TISER F1: {metrics['tiser_f1']:.2f}",
    )

print(
    "Macro Avg.",
    f"Direct EM: {summary['direct_macro_em']:.2f}",
    f"Direct F1: {summary['direct_macro_token_f1']:.2f}",
    f"TISER EM: {summary['tiser_macro_em']:.2f}",
    f"TISER F1: {summary['tiser_macro_token_f1']:.2f}",
)


tgqa_test Direct EM: 47.20 Direct F1: 64.41 TISER EM: 68.79 TISER F1: 80.95
tempreason_l2_test Direct EM: 89.93 Direct F1: 92.26 TISER EM: 90.32 TISER F1: 92.55
tempreason_l3_test Direct EM: 78.18 Direct F1: 82.09 TISER EM: 88.21 TISER F1: 90.58
timeqa_easy_test Direct EM: 92.46 Direct F1: 96.52 TISER EM: 94.43 TISER F1: 97.15
timeqa_hard_test Direct EM: 87.59 Direct F1: 92.44 TISER EM: 88.95 TISER F1: 92.40
Macro Avg. Direct EM: 79.07 Direct F1: 85.54 TISER EM: 86.14 TISER F1: 90.73


In [14]:
datasets = ["tgqa_test", "tempreason_l2_test", "tempreason_l3_test", "timeqa_easy_test", "timeqa_hard_test"]

for d in datasets:
    r = [x for x in records if x["dataset_name"] == d]
    n = len(r)

    direct_em = sum(x["direct_em"] for x in r) / n * 100
    direct_f1 = sum(x["direct_f1"] for x in r) / n * 100
    tiser_em = sum(x["tiser_em"] for x in r) / n * 100
    tiser_f1 = sum(x["tiser_f1"] for x in r) / n * 100

    print(
        f"{d}: "
        f"Direct EM={direct_em:.2f}, "
        f"Direct F1={direct_f1:.2f}, "
        f"TISER EM={tiser_em:.2f}, "
        f"TISER F1={tiser_f1:.2f}"
    )

tgqa_test: Direct EM=38.87, Direct F1=60.66, TISER EM=54.58, TISER F1=75.33
tempreason_l2_test: Direct EM=89.93, Direct F1=92.26, TISER EM=90.32, TISER F1=92.55
tempreason_l3_test: Direct EM=78.18, Direct F1=82.09, TISER EM=88.21, TISER F1=90.58
timeqa_easy_test: Direct EM=92.46, Direct F1=96.52, TISER EM=94.43, TISER F1=97.15
timeqa_hard_test: Direct EM=87.59, Direct F1=92.44, TISER EM=88.95, TISER F1=92.40
